In [1]:
import torch
import torch.nn as nn

In [3]:
a = [2.0, 1.0]  # 원본
b = [4.0, 2.0]  # 같은 방향
c = [-1.0, 2.0] # 직교
d = [-4.0, -2.0]# 반대방향

같은방향 = (torch.tensor(a), torch.tensor(b))
직교 = (torch.tensor(a), torch.tensor(c))
반대방향 = (torch.tensor(a), torch.tensor(d))

In [8]:
# 내적했을 때
print(torch.dot(같은방향[0], 같은방향[1]))  # 방향 양수
print(torch.dot(직교[0], 직교[1]))         # 0
print(torch.dot(반대방향[0], 반대방향[1]))  # 음수

# 내적은 방향성을 확인하는 점수, 채점표가 된다.
# 파라미터 학습을 하지 않아도. 행렬 곱 한번으로 방향성 확인.

tensor(10.)
tensor(0.)
tensor(-10.)


In [9]:

L, d = 6, 64
X_raw = torch.randn(L, d)

In [14]:
import math
# 전치행렬로 행렬곱
점수_raw = X_raw @ X_raw.T
어텐션_격자_raw = torch.softmax(점수_raw / math.sqrt(d), dim=1)

In [ ]:
#  대각선에 점수가 많이 매겨짐. => 자기 자신을 중요하게 봄.
# 대각선이 아닌 경우 점수가 낮다.
어텐션_격자_raw

tensor([[9.9739e-01, 2.8166e-04, 4.1118e-04, 5.8830e-04, 2.8921e-04, 1.0369e-03],
        [9.8743e-04, 9.9361e-01, 2.2699e-03, 6.9370e-04, 2.1208e-03, 3.1374e-04],
        [7.8000e-05, 1.2283e-04, 9.9931e-01, 2.2643e-05, 4.0583e-04, 5.6595e-05],
        [1.6416e-03, 5.5216e-04, 3.3307e-04, 9.9581e-01, 7.1389e-04, 9.4428e-04],
        [1.3342e-04, 2.7909e-04, 9.8695e-04, 1.1803e-04, 9.9834e-01, 1.3971e-04],
        [3.3044e-04, 2.8521e-05, 9.5078e-05, 1.0784e-04, 9.6512e-05, 9.9934e-01]])

In [17]:
Wq_t = nn.Linear(d, d, bias=False)
Wk_t = nn.Linear(d, d, bias=False)

In [23]:
torch.softmax(Wq_t(X_raw) @ Wk_t(X_raw).T / math.sqrt(d), dim= -1)

tensor([[0.1785, 0.1438, 0.1691, 0.1785, 0.1466, 0.1835],
        [0.1231, 0.1631, 0.1670, 0.1558, 0.1435, 0.2475],
        [0.2137, 0.1543, 0.1940, 0.1309, 0.2310, 0.0761],
        [0.1954, 0.1512, 0.1589, 0.1812, 0.1396, 0.1737],
        [0.1768, 0.2702, 0.1101, 0.1424, 0.1541, 0.1465],
        [0.1687, 0.1662, 0.1467, 0.1673, 0.2403, 0.1107]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# t셀프 어텐션
class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)   # Q 찾는 것
        self.Wk = nn.Linear(d, d, bias=False)   # K 이름
        self.Wv = nn.Linear(d, d, bias=False)   # V 내용
        self.Wo = nn.Linear(d, d, bias=False)   # O 출력
        
    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wk(x)

        d_k = K.size(-1)    # k의 차원수
        scores = Q @ K.T     # 점수 Q 와 K의 전치행렬 합성곱
        scores = scores / math.sqrt(d_k)

        # 가중치 : softmax 통과한 것
        weights = torch.softmax(scores, dim = 1)

        # 최종 출력 V 행렬 곱
        out = weights @ V
        return self.Wo(out), weights